# Working

**Request:** [T08] Fee calculation for August

> New email from Sai Suram <sai@agents.agentstore.it.com>
> Subject: [T08] Fee calculation for August
> Thread ID: AAQkADI1N2Y5MTE3LTE1MDctNGY0Yy1iYzQ5LWEzNmE5NzAyYzk4NQAQAGQ8zAq9DXVImCs5oO0bxns=
> 
> ## Sugges

Each cell below is one run in the sandbox, in the order it happened, with whatever it printed and produced.


In [1]:
import pandas as pd

transactions_raw = [
    ["Northwind", 42.00, "US", "no", "no"],
    ["Northwind", 3.50, "US", "no", "no"],
    ["Northwind", 128.00, "GB", "no", "no"],
    ["Northwind", 76.40, "US", "yes", "no"],
    ["Northwind", 250.00, "US", "no", "yes"],
    ["Contoso", 18.75, "US", "no", "no"],
    ["Contoso", 4.99, "DE", "no", "no"],
    ["Contoso", 310.00, "DE", "no", "no"],
    ["Contoso", 95.00, "US", "yes", "no"],
    ["Contoso", 62.30, "FR", "yes", "no"]
]

df = pd.DataFrame(transactions_raw, columns=['Merchant', 'Amount', 'CardCountry', 'CardPresent', 'Refunded'])
df['Amount'] = pd.to_numeric(df['Amount'])

# Monthly transaction counts (provided in the email)
monthly_counts = {
    'Northwind': 142,
    'Contoso': 64
}

def calculate_fee(row):
    amount = row['Amount']
    card_present = row['CardPresent'] == 'yes'
    international = row['CardCountry'] != 'US'
    refunded = row['Refunded'] == 'yes'
    merchant = row['Merchant']

    # Rule 5: Micro-transactions
    if amount < 5.00:
        fixed_fee = 0.15
        percentage_fee = 0.0
        total_fee = fixed_fee
    else:
        # Rule 4: Card-present transactions
        if card_present:
            fixed_fee = 0.10
            percentage_rate = 0.019  # 1.9%
        # Rule 2 & 1: Volume tier / Standard rate
        else:
            fixed_fee = 0.30
            if monthly_counts.get(merchant, 0) > 100:
                percentage_rate = 0.025 # 2.5% for volume tier
            else:
                percentage_rate = 0.029 # 2.9% for standard rate
        
        # Rule 3: International
        if international:
            percentage_rate += 0.010 # Add 1.0 percentage point
        
        percentage_fee = amount * percentage_rate
        total_fee = percentage_fee + fixed_fee

    # Rule 6: Refunds
    if refunded:
        # Only percentage component is returned
        total_fee -= percentage_fee 
        # Ensure fee doesn't go below fixed component for refunded transactions
        if total_fee < 0 and amount >= 5.00: # Only for non-micro transactions where fixed fee applies
            total_fee = fixed_fee # The fixed fee is not returned, so minimum fee for refund is fixed_fee
        elif total_fee < 0 and amount < 5.00: # For micro-transactions, fixed fee is 0.15, no percentage
            total_fee = 0.15

    # Final check for refunds - fixed fee is never returned. If percentage fee was the only component, then refund is 0.0
    # If the total fee became negative due to percentage refund, and there was a fixed fee, the fee should be the fixed fee.
    if refunded and not (amount < 5.00): # For non-micro transactions only.
        if card_present:
            total_fee = fixed_fee if (amount * percentage_rate) > 0 else 0.0 # If there was a percentage component, only fixed remains.
        else:
            total_fee = fixed_fee if (amount * percentage_rate) > 0 else 0.0
    elif refunded and amount < 5.00: # Micro-transactions refund
         total_fee = 0.15 # Fixed fee remains for micro-transactions.
    
    return total_fee


# Re-evaluating refund logic, it's tricky
def calculate_fee_revisited(row, monthly_counts):
    amount = row['Amount']
    card_present = row['CardPresent'] == 'yes'
    international = row['CardCountry'] != 'US'
    refunded = row['Refunded'] == 'yes'
    merchant = row['Merchant']

    fixed_fee_component = 0.0
    percentage_rate_component = 0.0

    # Rule 5: Micro-transactions (overrides all)
    if amount < 5.00:
        fixed_fee_component = 0.15
        percentage_rate_component = 0.0
    else:
        # Rule 4: Card-present
        if card_present:
            fixed_fee_component = 0.10
            percentage_rate_component = 0.019
        # Rule 1 & 2: Standard/Volume
        else:
            fixed_fee_component = 0.30
            if monthly_counts.get(merchant, 0) > 100:
                percentage_rate_component = 0.025
            else:
                percentage_rate_component = 0.029
        
        # Rule 3: International (adds to existing percentage rate)
        if international:
            percentage_rate_component += 0.010

    calculated_percentage_amount = amount * percentage_rate_component
    
    total_fee = calculated_percentage_amount + fixed_fee_component

    # Rule 6: Refunds - percentage component is returned, fixed component is not.
    if refunded:
        total_fee = fixed_fee_component # Only the fixed fee remains for refunded transactions
        # If a micro-transaction, fixed_fee_component is 0.15, so that's the refund fee
        # If it was a transaction with only percentage and no fixed fee (which isn't the case with these rules), it would be 0.

    return total_fee

df['CalculatedFee'] = df.apply(lambda row: calculate_fee_revisited(row, monthly_counts), axis=1)

# Summarize fees per merchant
merchant_summary = df.groupby('Merchant')['CalculatedFee'].sum().reset_index()
merchant_summary.rename(columns={'CalculatedFee': 'TotalFee'}, inplace=True)

print(df.to_json(orient='records'))
print(merchant_summary.to_json(orient='records'))

# Create an Excel writer object
output_filename = '/tmp/output/August_Fee_Calculation.xlsx'
writer = pd.ExcelWriter(output_filename, engine='xlsxwriter')

# Write the detailed transactions to one sheet
df.to_excel(writer, sheet_name='Transaction Details', index=False)

# Write the merchant summary to another sheet
merchant_summary.to_excel(writer, sheet_name='Merchant Summary', index=False)

# Close the Excel writer
writer.close()

[{"Merchant":"Northwind","Amount":42.0,"CardCountry":"US","CardPresent":"no","Refunded":"no","CalculatedFee":1.35},{"Merchant":"Northwind","Amount":3.5,"CardCountry":"US","CardPresent":"no","Refunded":"no","CalculatedFee":0.15},{"Merchant":"Northwind","Amount":128.0,"CardCountry":"GB","CardPresent":"no","Refunded":"no","CalculatedFee":4.78},{"Merchant":"Northwind","Amount":76.4,"CardCountry":"US","CardPresent":"yes","Refunded":"no","CalculatedFee":1.5516},{"Merchant":"Northwind","Amount":250.0,"CardCountry":"US","CardPresent":"no","Refunded":"yes","CalculatedFee":0.3},{"Merchant":"Contoso","Amount":18.75,"CardCountry":"US","CardPresent":"no","Refunded":"no","CalculatedFee":0.84375},{"Merchant":"Contoso","Amount":4.99,"CardCountry":"DE","CardPresent":"no","Refunded":"no","CalculatedFee":0.15},{"Merchant":"Contoso","Amount":310.0,"CardCountry":"DE","CardPresent":"no","Refunded":"no","CalculatedFee":12.39},{"Merchant":"Contoso","Amount":95.0,"CardCountry":"US","CardPresent":"yes","Refunde


[files written: August_Fee_Calculation.xlsx]
